In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from sklearn.metrics import precision_recall_fscore_support
import warnings
from pytorch_dataset import EBSDStrainDataset

class MultiModalEBSDStrainCNN(nn.Module):
    """
    Upgraded architecture that accepts both the EBSD image pattern
    and the crystal Euler angles to isolate strain effects accurately.
    """
    def __init__(self, euler_dim=3):
        super(MultiModalEBSDStrainCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.3)
        )

        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))

        # The input layer now accepts 128*4*4 features + the 3 Euler angles
        self.classifier = nn.Sequential(
            nn.Linear(128 * 4 * 4 + euler_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, eulers):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)

        # Concatenate Euler angles to the flattened CNN features
        x = torch.cat((x, eulers), dim=1)

        x = self.classifier(x)
        return x.squeeze()

def calculate_metrics(y_true, y_pred, strain_classes=None):
    epsilon = 1e-8
    percent_errors = np.abs((y_true - y_pred) / (y_true + epsilon)) * 100
    avg_percent_error = np.mean(percent_errors)
    std_dev_error = np.std(percent_errors)

    if strain_classes is None:
        strain_classes = np.sort(np.unique(y_true))

    strain_to_int_map = {strain: i for i, strain in enumerate(strain_classes)}

    y_pred_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - pred))] for pred in y_pred])
    y_true_snapped_floats = np.array([strain_classes[np.argmin(np.abs(strain_classes - true))] for true in y_true])

    y_pred_classes_int = np.array([strain_to_int_map[s] for s in y_pred_snapped_floats])
    y_true_classes_int = np.array([strain_to_int_map[s] for s in y_true_snapped_floats])

    warnings.filterwarnings('ignore')
    precision, recall, f1, _ = precision_recall_fscore_support(y_true_classes_int, y_pred_classes_int, average='weighted')

    return avg_percent_error, std_dev_error, precision, recall, f1

def train_and_evaluate(h5_path="ebsd_fcc_fe.h5", epochs=75, batch_size=128, max_lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    train_dataset = EBSDStrainDataset(h5_path=h5_path, split="train")
    test_dataset = EBSDStrainDataset(h5_path=h5_path, split="test")

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    model = MultiModalEBSDStrainCNN(euler_dim=3).to(device)
    criterion = nn.SmoothL1Loss()
    optimizer = optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-4)

    # OneCycleLR steps per batch rather than per epoch
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=max_lr, steps_per_epoch=len(train_loader), epochs=epochs
    )

    best_f1 = 0.0

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (patterns, strains, eulers) in enumerate(train_loader):
            # Explicitly cast eulers to float32 to match CNN feature types
            patterns = patterns.to(device)
            strains = strains.to(device)
            eulers = eulers.float().to(device)

            # Inject Gaussian Noise
            noise = torch.randn_like(patterns) * 0.05
            patterns = patterns + noise

            optimizer.zero_grad()

            # Pass both patterns and euler angles to the model
            outputs = model(patterns, eulers)

            loss = criterion(outputs, strains)
            loss.backward()
            optimizer.step()
            scheduler.step() # Step scheduler inside the batch loop

            running_loss += loss.item()

        model.eval()
        test_loss = 0.0
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for patterns, strains, eulers in test_loader:
                patterns = patterns.to(device)
                strains = strains.to(device)
                eulers = eulers.float().to(device)

                outputs = model(patterns, eulers)
                loss = criterion(outputs, strains)
                test_loss += loss.item()

                all_preds.extend(outputs.cpu().numpy())
                all_targets.extend(strains.cpu().numpy())

        test_loss /= len(test_loader)

        all_preds = np.array(all_preds)
        all_targets = np.array(all_targets)

        avg_pe, std_pe, precision, recall, f1 = calculate_metrics(all_targets, all_preds)

        print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {running_loss/len(train_loader):.4f} - Test Loss: {test_loss:.4f}")
        print(f"Metrics: P: {precision:.4f} | R: {recall:.4f} | F1: {f1:.4f} | % Error: {avg_pe:.2f}% (Std: {std_pe:.2f})")

        if f1 > best_f1:
            best_f1 = f1
            torch.save(model.state_dict(), "best_multimodal_ebsd_model.pth")

    print("\nTraining completed. Best model saved to 'best_multimodal_ebsd_model.pth'.")

if __name__ == "__main__":
    train_and_evaluate(h5_path="/content/drive/MyDrive/ebsd_fcc_fe.h5", epochs=75, batch_size=128)


Using device: cuda
Epoch [1/75] - Train Loss: 24.5934 - Test Loss: 18.9509
Metrics: P: 0.0156 | R: 0.1250 | F1: 0.0278 | % Error: 96.20% (Std: 12.37)
Epoch [2/75] - Train Loss: 24.2161 - Test Loss: 18.6153
Metrics: P: 0.0156 | R: 0.1250 | F1: 0.0278 | % Error: 91.85% (Std: 16.22)
Epoch [3/75] - Train Loss: 23.7927 - Test Loss: 18.2505
Metrics: P: 0.0156 | R: 0.1250 | F1: 0.0278 | % Error: 87.57% (Std: 18.40)
Epoch [4/75] - Train Loss: 23.2328 - Test Loss: 17.6569
Metrics: P: 0.0162 | R: 0.1250 | F1: 0.0287 | % Error: 81.63% (Std: 21.61)
Epoch [5/75] - Train Loss: 22.3697 - Test Loss: 16.7150
Metrics: P: 0.0318 | R: 0.1375 | F1: 0.0514 | % Error: 75.99% (Std: 23.27)
Epoch [6/75] - Train Loss: 20.9882 - Test Loss: 15.4543
Metrics: P: 0.0501 | R: 0.1383 | F1: 0.0733 | % Error: 72.70% (Std: 30.35)
Epoch [7/75] - Train Loss: 18.8012 - Test Loss: 12.7327
Metrics: P: 0.0922 | R: 0.1758 | F1: 0.1207 | % Error: 60.71% (Std: 30.65)
Epoch [8/75] - Train Loss: 15.1234 - Test Loss: 9.9872
Metrics: 